# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilip-Git18/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)


In [15]:
!git clone https://github.com/Dilip-Git18/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [16]:
!find . -name "content_refresh_anonymized.csv"

./flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [17]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("./flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [18]:
# Create target label

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [19]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


# 1. Ranked actions + reason codes

The purpose of this playbook is to convert the machine learning model output into practical actions that a content manager can review. The recommendations are intended to help prioritize content that may benefit from updates while keeping a human reviewer in the decision loop.

## Action Priority

| Priority | Recommended Action | Reason Code |
|-----------|--------------------|-------------|
| High | Refresh content immediately | Declining traffic with high historical visibility |
| High | Update SEO title and meta description | Low CTR despite strong impressions |
| Medium | Improve content quality and keyword coverage | Moderate traffic decline |
| Medium | Review search intent alignment | Engagement metrics decreasing |
| Low | Continue monitoring | Stable performance with no strong decline |

The recommendations are designed as decision-support rather than automatic actions.## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [20]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


In [21]:
# ======================================================
# Build Ranked Content Action Queue
# ======================================================

queue = df.copy()

# Priority

queue["Priority"] = np.where(
    (queue["is_declining_label"] == 1) &
    (queue["impressions_90d"] > queue["impressions_90d"].median()),
    "High",
    np.where(
        queue["is_declining_label"] == 1,
        "Medium",
        "Low"
    )
)

# Reason Codes

queue["Reason Code"] = "Monitor"

queue.loc[
    queue["ctr"] < 1,
    "Reason Code"
] = "Low CTR"

queue.loc[
    queue["engagement_rate"] < queue["engagement_rate"].median(),
    "Reason Code"
] = "Low Engagement"

queue.loc[
    queue["is_declining_label"] == 1,
    "Reason Code"
] = "Traffic Decline"

# Recommended Actions

queue["Recommended Action"] = np.select(
    [
        queue["Priority"] == "High",
        queue["Priority"] == "Medium",
        queue["Priority"] == "Low"
    ],
    [
        "Refresh Immediately",
        "Review and Update",
        "Monitor"
    ],
    default="Monitor"
)

# Rank

priority_order = {
    "High":0,
    "Medium":1,
    "Low":2
}

queue["Rank"] = queue["Priority"].map(priority_order)

queue = queue.sort_values(
    ["Rank","impressions_90d"],
    ascending=[True,False]
)

queue = queue.reset_index(drop=True)

queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,Priority,Reason Code,Recommended Action,Rank
0,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.38,excellent,page_1,down,-44.8,1,High,Traffic Decline,Refresh Immediately,0
1,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,0.00,excellent,top_3,down,-44.5,1,High,Traffic Decline,Refresh Immediately,0
2,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,0.00,excellent,top_3,down,-33.2,1,High,Traffic Decline,Refresh Immediately,0
3,content_1a9e894be2e2,client_19581e27de,70.0,0.07,LOW,0.10,keyword article,transactional,NaN,NaN,...,0.00,excellent,page_1,down,-27.0,1,High,Traffic Decline,Refresh Immediately,0
4,content_2c2606c5d176,client_19581e27de,590.0,0.18,LOW,0.31,keyword article,commercial,NaN,NaN,...,0.00,excellent,page_1,down,-36.5,1,High,Traffic Decline,Refresh Immediately,0
5,content_cb112fce36be,client_19581e27de,70.0,0.65,MEDIUM,0.34,keyword article,transactional,2761.0,18472.0,...,0.00,excellent,page_1,down,-41.8,1,High,Traffic Decline,Refresh Immediately,0
6,content_9532f197bbc8,client_4e07408562,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,excellent,top_3,down,-37.3,1,High,Traffic Decline,Refresh Immediately,0
7,content_008fb02c46cb,client_349c41201b,10.0,0.86,HIGH,0.00,keyword article,commercial,3338.0,21685.0,...,0.00,excellent,page_1,down,-21.3,1,High,Traffic Decline,Refresh Immediately,0
8,content_813e88069237,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,commercial,4610.0,30146.0,...,0.52,excellent,page_3_5,down,-33.8,1,High,Traffic Decline,Refresh Immediately,0
9,content_ff94c9b6b411,client_349c41201b,20.0,0.00,LOW,0.00,keyword article,informational,5375.0,35334.0,...,1.15,excellent,page_3_5,down,-32.4,1,High,Traffic Decline,Refresh Immediately,0


# 2. Intended Use and Limits

## Intended Use

This content action playbook is designed to support SEO specialists and content managers in prioritizing which content should be reviewed for updates. The ranked recommendations are based on historical performance data and observed model predictions.

The playbook is intended as a **decision-support tool** rather than an automated decision-making system.

## Limits

- The recommendations are based only on the available historical dataset.
- Future search trends and algorithm updates are not included.
- Performance may differ for new clients or different datasets.
- Human review is required before any content changes are made.
- The model should be used to prioritize review, not to replace editorial judgement.

# 3. Human Review + the No-Go List

## Human Review

Before acting on any recommendation, a content manager should verify:

- The content still matches the current search intent.
- Keywords are relevant and up to date.
- Facts and information are accurate.
- SEO title and meta description are appropriate.
- Internal and external links are working.
- Business priorities support updating the content.

The model provides recommendations only. Final decisions should always be made by a human reviewer.

## No-Go List

The following actions should **never** be automated:

- Publishing updated content automatically.
- Deleting content pages.
- Changing URLs or redirects.
- Updating client-specific business information.
- Making business decisions without human approval.

The playbook is intended only as a decision-support tool.

# 4. Monitoring / Retrain Triggers

The model should be monitored regularly to ensure the recommendations remain useful.

Suggested retraining triggers include:

- A noticeable decrease in prediction performance.
- New clients or a large amount of new content.
- Changes in search engine behaviour or SEO trends.
- Significant drift in feature distributions.
- Scheduled retraining every 3–6 months.

Regular monitoring helps maintain reliable recommendations over time.

In [22]:
# ==========================================
# Export files for the research paper
# ==========================================

import os

os.makedirs("work/outputs", exist_ok=True)

export_columns = [
    "content_id",
    "Priority",
    "Recommended Action",
    "Reason Code",
    "impressions_90d",
    "ctr",
    "is_declining_label"
]

queue[export_columns].to_csv(
    "work/outputs/content_action_queue.csv",
    index=False
)

summary = (
    queue.groupby("Priority")
         .size()
         .reset_index(name="Count")
)

summary.to_csv(
    "work/outputs/action_summary.csv",
    index=False
)

print("Export completed successfully!")

summary

Export completed successfully!


,Priority,Count
0,High,8906
1,Low,13738
2,Medium,7356


In [23]:
!ls work/outputs

action_summary.csv  content_action_queue.csv


# 5. Exports for the Paper

The ranked content action queue has been exported for use in the final research paper.

## Exported Files

| File | Purpose |
|------|---------|
| **content_action_queue.csv** | Contains the ranked list of content items with priority levels, recommended actions, and reason codes. This file will be used as the primary recommendation table in the research paper. |
| **action_summary.csv** | Provides a summary of the number of content items in each priority category (High, Medium, and Low). This file can be reused for tables or charts in the paper. |

These exports are intended to support the research paper and should be interpreted as **decision-support** outputs rather than fully automated recommendations. Any actions based on these files should be reviewed by a human before implementation.

# Self-check

- ✅ Every section above is filled with both markdown explanations and executable code where appropriate.
- ✅ The notebook runs successfully from top to bottom without errors.
- ✅ No client names, private URLs, or confidential information are included.
- ✅ All recommendations use careful language such as **observed**, **measured**, **directional**, and **decision-support**.
- ✅ The ranked content action queue and summary files have been exported to `work/outputs/`.
- ✅ The notebook has been committed under `work/notebooks/w07_action_playbook.ipynb` and is ready for submission.